In [ ]:
# RICARDO DOMINGUEZ ALCALA
# JORGE NATANAEL CASTOLO GONZALEZ
# DIEGO EDUARDO SILVA MONTES

In [ ]:
!nvidia-smi


In [2]:

%%writefile generador.cu

#include <iostream>
#include <fstream>
#include <vector>
#include <cuda_runtime.h>
#include <curand_kernel.h>

using namespace std;

const int N = 10000001;

#define CUDA_CHECK(call)                                      \
    do {                                                      \
        cudaError_t error = call;                             \
        if (error != cudaSuccess) {                           \
            cerr << cudaGetErrorString(error) << endl;         \
            exit(EXIT_FAILURE);                               \
        }                                                     \
    } while (0)

__global__ void generarDatos(int *arreglo, int size)
{
    int id = blockIdx.x * blockDim.x + threadIdx.x;

    if (id < size)
    {
        curandState estado;

        curand_init(12345ULL + id, 0, 0, &estado);

        arreglo[id] = curand(&estado) % 1000001;
    }
}

int main()
{
    vector<int> arreglo(N);

    int *d_arreglo;

    CUDA_CHECK(cudaMalloc(&d_arreglo, N * sizeof(int)));

    int hilos = 256;
    int bloques = (N + hilos - 1) / hilos;

    generarDatos<<<bloques, hilos>>>(d_arreglo, N);

    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    CUDA_CHECK(cudaMemcpy(
        arreglo.data(),
        d_arreglo,
        N * sizeof(int),
        cudaMemcpyDeviceToHost
    ));

    ofstream archivo("10_millones_numeros.txt");

    if (!archivo.is_open())
    {
        cerr << "Error al crear el archivo" << endl;
        cudaFree(d_arreglo);
        return 1;
    }

    for (int numero : arreglo)
    {
        archivo << numero << '\n';
    }

    archivo.close();

    CUDA_CHECK(cudaFree(d_arreglo));

    cout << "====================================" << endl;
    cout << " GENERACION PARALELA DE DATOS CUDA" << endl;
    cout << "====================================" << endl;
    cout << "Archivo generado correctamente" << endl;
    cout << "Total de numeros: " << N << endl;
    cout << "Rango: 0 a 1000000" << endl;
    cout << "Bloques utilizados: " << bloques << endl;
    cout << "Hilos por bloque: " << hilos << endl;
    cout << "Archivo: 10_millones_numeros.txt" << endl;
    cout << "====================================" << endl;

    return 0;
}


Overwriting generador.cu


In [10]:
!nvcc -O3 -arch=sm_75 generador.cu -o generador

In [4]:
!./generador

 GENERACION PARALELA DE DATOS CUDA
Archivo generado correctamente
Total de numeros: 10000001
Rango: 0 a 1000000
Bloques utilizados: 39063
Hilos por bloque: 256
Archivo: 10_millones_numeros.txt


In [8]:
!wget -O 10_millones_numeros.txt "https://raw.githubusercontent.com/RicardoDomi/Problema-pr-ctico-en-OpenMP/main/Proyecto_final%3AProblema_4/10_millones_numeros.txt"

--2026-11-26 00:31:19--  https://raw.githubusercontent.com/RicardoDomi/Problema-pr-ctico-en-OpenMP/main/Proyecto_final%3AProblema_4/10_millones_numeros.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 68890903 (66M) [text/plain]
Saving to: ‘10_millones_numeros.txt’

10_millones_numeros 100%[===================>]  65.70M   217MB/s    in 0.3s    

2026-11-26 00:31:21 (217 MB/s) - ‘10_millones_numeros.txt’ saved [68890903/68890903]



In [17]:
!head -n 5 10_millones_numeros.txt
!wc -l 10_millones_numeros.txt
!sha256sum 10_millones_numeros.txt

473896
869929
926024
648304
214656
10000001 10_millones_numeros.txt
d196c862346c11f190424798e3dbad52144a60c4e2e92663fd884e343de3cc8d  10_millones_numeros.txt


In [13]:

%%writefile percentiles_cuda.cu

#include <iostream>
#include <fstream>
#include <chrono>
#include <stdio.h>
#include <cstdlib>
#include <cuda_runtime.h>
#include <thrust/device_ptr.h>
#include <thrust/sort.h>

using namespace std;

bool ordenado = false;

#define CUDA_CHECK(call)                                      \
    do {                                                      \
        cudaError_t error = call;                             \
        if (error != cudaSuccess) {                           \
            cerr << cudaGetErrorString(error) << endl;         \
            exit(EXIT_FAILURE);                               \
        }                                                     \
    } while (0)

// Mostrar los primeros 20 elementos del arreglo

void mostrarArreglo(int *arreglo, int size)
{
    int limite = (size < 20) ? size : 20;

    for (int i = 0; i < limite; i++)
    {
        printf("%d ", arreglo[i]);
    }

    printf("\n");
}

// Calcular un percentil mediante GPU
// Misma formula que Secuencial y OpenMP

__device__ double calcularPercentil(
    const int *arreglo,
    int size,
    int k)
{
    double posicion = (k * (size + 1.0)) / 100.0;

    if (posicion <= 1)
        return arreglo[0];

    if (posicion >= size)
        return arreglo[size - 1];

    int indice = (int)posicion;
    double decimal = posicion - indice;

    return arreglo[indice - 1] +
           decimal * (arreglo[indice] - arreglo[indice - 1]);
}

// Kernel para calcular los seis percentiles

__global__ void calcularPercentiles(
    const int *arreglo,
    int size,
    double *resultados)
{
    int id = blockIdx.x * blockDim.x + threadIdx.x;

    const int percentiles[] = {10, 25, 50, 75, 90, 95};

    if (id < 6)
    {
        resultados[id] = calcularPercentil(
            arreglo, size, percentiles[id]);
    }
}

// Ordenamiento paralelo mediante CUDA

void ordenarParalelo(int *d_arreglo, int N)
{
    thrust::device_ptr<int> ptr(d_arreglo);

    thrust::sort(ptr, ptr + N);

    CUDA_CHECK(cudaDeviceSynchronize());
}

int main()
{
    // Informacion del hardware

    cudaDeviceProp propiedades;

    CUDA_CHECK(cudaGetDeviceProperties(&propiedades, 0));

    cout << "====================================" << endl;
    cout << " CALCULO DE PERCENTILES - CUDA" << endl;
    cout << "====================================" << endl;
    cout << "GPU: " << propiedades.name << endl;
    cout << "Multiprocesadores: "
         << propiedades.multiProcessorCount << endl;

    // Lectura del archivo

    ifstream archivo("10_millones_numeros.txt");

    if (!archivo.is_open())
    {
        cout << "Error al abrir el archivo" << endl;
        return 1;
    }

    cout << "Archivo abierto correctamente" << endl;

    int N = 0;
    int numero;

    while (archivo >> numero)
    {
        if (numero < 0 || numero > 1000000)
        {
            cout << "Error: numero fuera de rango" << endl;
            return 1;
        }

        N++;
    }

    if (!archivo.eof())
    {
        cout << "Error: el archivo contiene datos invalidos" << endl;
        return 1;
    }

    if (N <= 10000000)
    {
        cout << "Error: cantidad de datos insuficiente" << endl;
        return 1;
    }

    cout << "Total de numeros: " << N << endl;

    archivo.clear();
    archivo.seekg(0, ios::beg);

    int *arreglo = new int[N];

    for (int i = 0; i < N; i++)
    {
        archivo >> arreglo[i];
    }

    archivo.close();

    // Reservar memoria en GPU

    int *d_arreglo;
    double *d_resultados;

    CUDA_CHECK(cudaMalloc(
        &d_arreglo, N * sizeof(int)));

    CUDA_CHECK(cudaMalloc(
        &d_resultados, 6 * sizeof(double)));

    int opcion;
    int percentiles[] = {10, 25, 50, 75, 90, 95};

    double resultados[6];
    double RIC = 0.0;
    double tiempoProcesamiento = 0.0;

    do
    {
        printf("\n========== MENU ==========\n");
        printf("1. Mostrar arreglo\n");
        printf("2. Ordenar de menor a mayor\n");
        printf("3. Calcular los percentiles\n");
        printf("4. Calcular los cuartiles\n");
        printf("5. Mostrar el valor correspondiente a cada percentil y cuartil\n");
        printf("6. Calcular el rango intercuartilico (RIC)\n");
        printf("7. Salir\n");
        printf("==========================\n");
        printf("Selecciona una opcion: ");

        if (scanf("%d", &opcion) != 1)
        {
            printf("\nEntrada finalizada o invalida\n");
            break;
        }

        switch (opcion)
        {
            case 1:
                mostrarArreglo(arreglo, N);
                break;

            case 2:
            {
                if (ordenado)
                {
                    printf("El arreglo ya esta ordenado\n");
                    break;
                }

                // Inicio del cronometro

                auto inicio = chrono::steady_clock::now();

                // Transferir datos de CPU a GPU

                CUDA_CHECK(cudaMemcpy(
                    d_arreglo,
                    arreglo,
                    N * sizeof(int),
                    cudaMemcpyHostToDevice
                ));

                // Ordenar mediante GPU

                ordenarParalelo(d_arreglo, N);

                // Calcular percentiles mediante GPU

                calcularPercentiles<<<1, 32>>>(
                    d_arreglo,
                    N,
                    d_resultados
                );

                CUDA_CHECK(cudaGetLastError());
                CUDA_CHECK(cudaDeviceSynchronize());

                // Recuperar los percentiles

                CUDA_CHECK(cudaMemcpy(
                    resultados,
                    d_resultados,
                    6 * sizeof(double),
                    cudaMemcpyDeviceToHost
                ));

                // Calcular rango intercuartilico

                RIC = resultados[3] - resultados[1];

                // Fin del cronometro

                auto fin = chrono::steady_clock::now();

                chrono::duration<double> tiempo = fin - inicio;

                tiempoProcesamiento = tiempo.count();

                // Recuperar arreglo ordenado para el menu
                // Esta copia no es necesaria para obtener
                // los percentiles, por eso queda fuera
                // del tiempo de procesamiento.

                CUDA_CHECK(cudaMemcpy(
                    arreglo,
                    d_arreglo,
                    N * sizeof(int),
                    cudaMemcpyDeviceToHost
                ));

                ordenado = true;

                printf("Procesamiento completado\n");
                printf("Tiempo: %.6f segundos\n",
                       tiempoProcesamiento);

                break;
            }

            case 3:
                if (ordenado)
                {
                    for (int i = 0; i < 6; i++)
                    {
                        printf("P%d = %.2f\n",
                               percentiles[i],
                               resultados[i]);
                    }
                }
                else
                {
                    printf("Primero debes ordenar el arreglo\n");
                }
                break;

            case 4:
                if (ordenado)
                {
                    printf("Q1 = %.2f\n", resultados[1]);
                    printf("Q2 = %.2f\n", resultados[2]);
                    printf("Q3 = %.2f\n", resultados[3]);
                }
                else
                {
                    printf("Primero debes ordenar el arreglo\n");
                }
                break;

            case 5:
                if (ordenado)
                {
                    printf("\n===== PERCENTILES =====\n");

                    for (int i = 0; i < 6; i++)
                    {
                        printf("P%d = %.2f\n",
                               percentiles[i],
                               resultados[i]);
                    }

                    printf("\n===== CUARTILES =====\n");

                    printf("Q1 = %.2f\n", resultados[1]);
                    printf("Q2 = %.2f\n", resultados[2]);
                    printf("Q3 = %.2f\n", resultados[3]);

                    printf("\nTiempo: %.6f segundos\n",
                           tiempoProcesamiento);
                }
                else
                {
                    printf("Primero debes ordenar el arreglo\n");
                }
                break;

            case 6:
                if (ordenado)
                {
                    printf("RIC = %.2f\n", RIC);
                }
                else
                {
                    printf("Primero debes ordenar el arreglo\n");
                }
                break;

            case 7:
                printf("Saliendo...\n");
                break;

            default:
                printf("Opcion no valida\n");
        }

    } while (opcion != 7);

    // Liberar memoria

    CUDA_CHECK(cudaFree(d_arreglo));
    CUDA_CHECK(cudaFree(d_resultados));

    delete[] arreglo;

    return 0;
}


Overwriting percentiles_cuda.cu


In [14]:
!nvcc -O3 -arch=sm_75 percentiles_cuda.cu -o percentiles_cuda

In [15]:
!printf "2\n5\n6\n7\n" | ./percentiles_cuda

 CALCULO DE PERCENTILES - CUDA
GPU: Tesla T4
Multiprocesadores: 40
Archivo abierto correctamente
Total de numeros: 10000001

========== MENU ==========
1. Mostrar arreglo
2. Ordenar de menor a mayor
3. Calcular los percentiles
4. Calcular los cuartiles
5. Mostrar el valor correspondiente a cada percentil y cuartil
6. Calcular el rango intercuartilico (RIC)
7. Salir
Selecciona una opcion: Procesamiento completado
Tiempo: 0.015472 segundos

========== MENU ==========
1. Mostrar arreglo
2. Ordenar de menor a mayor
3. Calcular los percentiles
4. Calcular los cuartiles
5. Mostrar el valor correspondiente a cada percentil y cuartil
6. Calcular el rango intercuartilico (RIC)
7. Salir
Selecciona una opcion: 
===== PERCENTILES =====
P10 = 100157.00
P25 = 250171.00
P50 = 500286.00
P75 = 750225.00
P90 = 900042.00
P95 = 950070.90

===== CUARTILES =====
Q1 = 250171.00
Q2 = 500286.00
Q3 = 750225.00

Tiempo: 0.015472 segundos

========== MENU ==========
1. Mostrar arreglo
2. Ordenar de menor a mayor


In [16]:
import numpy as np

datos = np.loadtxt(
    "10_millones_numeros.txt",
    dtype=np.int32
)

percentiles = np.percentile(
    datos,
    [10, 25, 50, 75, 90, 95],
    method="weibull"
)

print("===== VALIDACION INDEPENDIENTE =====")
print("Total de datos:", len(datos))
print("Minimo:", datos.min())
print("Maximo:", datos.max())

print("\n===== PERCENTILES =====")

for p, valor in zip([10, 25, 50, 75, 90, 95], percentiles):
    print(f"P{p} = {valor:.2f}")

print("\n===== CUARTILES =====")

print(f"Q1 = {percentiles[1]:.2f}")
print(f"Q2 = {percentiles[2]:.2f}")
print(f"Q3 = {percentiles[3]:.2f}")

print(f"\nRIC = {percentiles[3] - percentiles[1]:.2f}")

===== VALIDACION INDEPENDIENTE =====
Total de datos: 10000001
Minimo: 0
Maximo: 1000000

===== PERCENTILES =====
P10 = 100157.00
P25 = 250171.00
P50 = 500286.00
P75 = 750225.00
P90 = 900042.00
P95 = 950070.90

===== CUARTILES =====
Q1 = 250171.00
Q2 = 500286.00
Q3 = 750225.00

RIC = 500054.00
